# B3. Chia tập huấn luyện / kiểm tra

`application_train` → 80/20, **ngẫu nhiên phân tầng theo `TARGET`, seed 42**. Lưu danh sách `SK_ID_CURR` mỗi tập ở `ml/data/split.csv`
(cột `SK_ID_CURR, tap`); mọi notebook sau đọc tập từ file này.

- Không chia theo thời gian được: mọi cột thời gian là số ngày/tháng tính ngược từ ngày nộp của chính hồ sơ, dữ liệu không có ngày nộp.
- Phân tầng: chia riêng từng lớp theo đúng 80/20 nên hai tập có cùng tỷ lệ `TARGET` = 1.
- Chia **trước** EDA, chọn đặc trưng, tinh chỉnh: tập test đóng vai dữ liệu tương lai, chỉ chạm một lần ở B11.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
RAW_DIR = ROOT / "data" / "ml" / "home_credit"
DATA_DIR = ROOT / "ml" / "data"

import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split

SEED, TEST_SIZE = 42, 0.2
SPLIT_PATH = DATA_DIR / "split.csv"

In [2]:
a = pd.read_csv(RAW_DIR / "application_train.csv", usecols=["SK_ID_CURR", "TARGET", "NAME_CONTRACT_TYPE"])
tr, te = train_test_split(a.SK_ID_CURR, test_size=TEST_SIZE, stratify=a.TARGET, random_state=SEED)
s = pd.concat([pd.DataFrame({"SK_ID_CURR": tr, "tap": "train"}), pd.DataFrame({"SK_ID_CURR": te, "tap": "test"})]).sort_values("SK_ID_CURR")
s.to_csv(SPLIT_PATH, index=False)
assert len(s) == len(a) and s.SK_ID_CURR.is_unique
print("md5", hashlib.md5(SPLIT_PATH.read_bytes()).hexdigest())

md5 ae85fe67e2d3ae8b922e7a9a8470c6cf


In [3]:
g = a.merge(s, on="SK_ID_CURR").groupby("tap", sort=False)
pd.DataFrame({"số hồ sơ": g.size(), "TARGET = 1": g.TARGET.sum(), "tỷ lệ TARGET = 1": g.TARGET.mean(),
              "tỷ lệ Revolving (AT Flex)": g.NAME_CONTRACT_TYPE.apply(lambda v: (v == "Revolving loans").mean())})

,số hồ sơ,TARGET = 1,tỷ lệ TARGET = 1,tỷ lệ Revolving (AT Flex)
tap,,,,
train,246008,19860,0.080729,0.095314
test,61503,4965,0.080728,0.094808
